In [10]:
import os
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)



In [11]:
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"

In [12]:
# Load splits
X_train = pd.read_pickle(os.path.join(PROCESSED_DIR, "X_train.pkl"))
X_test  = pd.read_pickle(os.path.join(PROCESSED_DIR, "X_test.pkl"))
y_train = pd.read_pickle(os.path.join(PROCESSED_DIR, "y_train.pkl"))
y_test  = pd.read_pickle(os.path.join(PROCESSED_DIR, "y_test.pkl"))


In [13]:
# Load preprocessing artifacts 
artifacts = joblib.load(os.path.join(MODELS_DIR, "preprocessing_artifacts.pkl"))
target_col = artifacts["target_col"]
feature_columns = artifacts["feature_columns"]

In [14]:
# Sanity checks 
assert list(X_train.columns) == feature_columns, "X_train columns != saved feature order"
assert list(X_test.columns) == feature_columns, "X_test columns != saved feature order"
assert len(X_train) == len(y_train), "X_train/y_train length mismatch"
assert len(X_test) == len(y_test), "X_test/y_test length mismatch"

print("Loaded splits (dataset-agnostic):")
print("  X_train:", X_train.shape, "| X_test:", X_test.shape)
print("  target :", target_col)
print("  features:", len(feature_columns))
print("  class distribution (train):", y_train.value_counts().to_dict())

Loaded splits (dataset-agnostic):
  X_train: (12000, 31) | X_test: (3000, 31)
  target : Diagnosis
  features: 31
  class distribution (train): {0: 6026, 1: 5974}


In [15]:


IMBALANCE_THRESHOLD = 0.40

def detect_imbalance(y, threshold=IMBALANCE_THRESHOLD):
    counts = y.value_counts()
    props = y.value_counts(normalize=True)
    minority_share = props.min()          

    is_imbalanced = minority_share < threshold
    class_weight = "balanced" if is_imbalanced else None

    return {
        "class_counts": counts.to_dict(),
        "class_proportions": props.round(3).to_dict(),
        "n_classes": len(counts),
        "minority_share": round(minority_share, 3),
        "threshold": threshold,
        "is_imbalanced": is_imbalanced,
        "class_weight": class_weight,
    }

imbalance_report = detect_imbalance(y_train)

print("Class counts     :", imbalance_report["class_counts"])
print("Class proportions:", imbalance_report["class_proportions"])
print("Num classes      :", imbalance_report["n_classes"])
print("Minority share   :", imbalance_report["minority_share"])
print(f"Imbalanced (<{IMBALANCE_THRESHOLD}): ", imbalance_report["is_imbalanced"])
print("class_weight     :", imbalance_report["class_weight"])

Class counts     : {0: 6026, 1: 5974}
Class proportions: {0: 0.502, 1: 0.498}
Num classes      : 2
Minority share   : 0.498
Imbalanced (<0.4):  False
class_weight     : None


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE = 42

def build_models(class_weight=None, random_state=RANDOM_STATE):
    
    return {
        "LogisticRegression": LogisticRegression(
            class_weight=class_weight, random_state=random_state, max_iter=1000
        ),
        "DecisionTree": DecisionTreeClassifier(
            class_weight=class_weight, random_state=random_state
        ),
        "RandomForest": RandomForestClassifier(
            class_weight=class_weight, random_state=random_state
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state
        ),
    }

models = build_models(class_weight=imbalance_report["class_weight"])

print("Built models (class_weight =", imbalance_report["class_weight"], "):")
for name, model in models.items():
    print(f"\n  {name}:")
    print("   ", model)

Built models (class_weight = None ):

  LogisticRegression:
    LogisticRegression(max_iter=1000, random_state=42)

  DecisionTree:
    DecisionTreeClassifier(random_state=42)

  RandomForest:
    RandomForestClassifier(random_state=42)

  GradientBoosting:
    GradientBoostingClassifier(random_state=42)


In [17]:


from sklearn.utils.class_weight import compute_sample_weight


gb_sample_weight = None
if imbalance_report["class_weight"] == "balanced":
    gb_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

trained_models = {}
training_report = {}

for name, model in models.items():
    try:
        if name == "GradientBoosting" and gb_sample_weight is not None:
            model.fit(X_train, y_train, sample_weight=gb_sample_weight)
            weight_status = "sample_weight=balanced"
        elif name == "GradientBoosting":
            model.fit(X_train, y_train)
            weight_status = "none"
        else:
            model.fit(X_train, y_train)
            weight_status = f"class_weight={imbalance_report['class_weight']}"

        trained_models[name] = model
        training_report[name] = {"trained": True, "weighting": weight_status}
    except Exception as e:
        training_report[name] = {"trained": False, "error": str(e)}

print("Training complete:")
for name, info in training_report.items():
    print(f"  {name:20s} | {info}")

Training complete:
  LogisticRegression   | {'trained': True, 'weighting': 'class_weight=None'}
  DecisionTree         | {'trained': True, 'weighting': 'class_weight=None'}
  RandomForest         | {'trained': True, 'weighting': 'class_weight=None'}
  GradientBoosting     | {'trained': True, 'weighting': 'none'}
